# RAG 과제

## Step-0. 환경 세팅

In [ ]:
%pwd

In [ ]:
import os
os.environ["RAG_DATA_DIR"] = "/content/data"

from google.colab import drive; drive.mount("/content/drive")

REPO = "/content/drive/MyDrive/RAG-assignment"  # 본인이 압축 푼 폴더 경로
%cd {REPO}
!pip install -q -e .

import sys, types, importlib
sys.path.insert(0, f"{REPO}/src")
sys.modules.setdefault("imp", types.SimpleNamespace(reload=importlib.reload))
%load_ext autoreload
%autoreload 2

In [ ]:
%pwd

In [ ]:
!sudo apt-get update -q
!sudo apt install tree -q

In [ ]:
# 폴더 구조 체크
!tree -I '__pycache__|*.egg-info' .

## Step-1. 데이터 다운로드

In [ ]:
from huggingface_hub import snapshot_download

from rag_assignment.paths import DATA_DIR


REPO_ID = "NLPlab-skku/RAG-assignment"
snapshot_download("NLPlab-skku/RAG-assignment", repo_type="dataset",
                  revision="v1", local_dir=DATA_DIR)

## Step-2. Config 설정

In [ ]:
import re

from rag_assignment.paths import OUTPUTS_DIR

def _get_model_name(model_name_or_path: str) -> str:
    name = model_name_or_path.rstrip("/").split("/")[-1]
    return re.sub(r"-Instruct.*$", "", name)

### Retriever
RETRIEVER_MODE = "dense"            # "lexical" or "dense"
RETRIEVAL_K = 100

### Generator
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
CONTEXT_K = 5
USE_CONTEXT = True
MAX_SEQ_LEN = 2048
MAX_ANS_LEN = 32
BATCH_SIZE = 8
USE_FP16 = True

### Result
model_name = _get_model_name(GENERATOR_MODEL)
OUTPUT_DIR = OUTPUTS_DIR / f"{RETRIEVER_MODE}_Rk={RETRIEVAL_K}_Ck={CONTEXT_K}_{USE_CONTEXT}_{model_name}"

## Step-3. Module 준비
- Retriever와 Generator 준비

### Load retriever

In [ ]:
from rag_assignment.utils import load_corpus
from rag_assignment.paths import DATA_DIR, INDEX_DIR

corpus = load_corpus(DATA_DIR / "corpus.jsonl")
index_dir = INDEX_DIR / RETRIEVER_MODE

In [ ]:
from rag_assignment.retriever import get_retriever

retriever = get_retriever(RETRIEVER_MODE, corpus, index_dir)

### Load generator

In [ ]:
from rag_assignment.generator import HFGenerator

generator = HFGenerator(
    model_name_or_path=GENERATOR_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    max_ans_length=MAX_ANS_LEN,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16
)

## Step-4. Warmup
* 초반 시간 측정의 부정확함 방지

### Load Dev QA

In [ ]:
from rag_assignment.utils import load_qa_dataset

dev_qa = load_qa_dataset(DATA_DIR / "dev_qa.jsonl")
qa = dev_qa[0]
docs = [r.doc for r in retriever.retrieve(qa.question, CONTEXT_K)]
for d in docs:
    print(d.pid, d.title, "|", d.text[:80])
out = generator.generate(qa.question, docs)

print("Q:", qa.question, "| gold:", qa.answers)
print("raw:", repr(out.raw_text), "-> pred:", out.text)

## Step-5. 실행

In [ ]:
from rag_assignment.pipeline import RAGPipeline

pipeline = RAGPipeline(retriever, generator, RETRIEVAL_K, CONTEXT_K)
pipeline.run(dev_qa, use_context=USE_CONTEXT, output_dir=OUTPUT_DIR)

## Step-6. 평가

In [ ]:
from rag_assignment.grading.evaluate import evaluate, print_summary

summary, results = evaluate(OUTPUT_DIR, dev_qa, corpus)
print_summary(summary)

## Step-7. 오답 분석 (Optional)
- 확인용 오답 분석

In [ ]:
hit_but_wrong = [r for r in results if r.metrics.recall_at_k[CONTEXT_K] and not r.metrics.span_em]
miss = [r for r in results if not r.metrics.recall_at_k[CONTEXT_K]]
print(f"검색 성공·생성 실패 {len(hit_but_wrong)} / 검색 실패 {len(miss)}")
for r in hit_but_wrong[:5]:
    print(r.question, "| pred:", r.pred_answer, "| gold:", r.answers)